# Flow over a NACA 0012 Airfoil

## Import Library

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Domain Setup

In [2]:
C      = 1.0   # Chord length
AoA    = 5.0   # Angle of attack (degrees)
alpha  = np.deg2rad(AoA)
t_naca = 0.12  # NACA 4-digit thickness ratio → NACA 0012

Lx = 22  # Domain length in x (22 chords)
Ly = 10  # Domain height in y (10 chords)

Nx = 220  # Grid points in x (10 pts per chord)
Ny = 100  # Grid points in y

dx = Lx / Nx
dy = Ly / Ny

x_le = 5.0      # Leading-edge x position (5 chords from inlet)
y_c  = Ly / 2   # Chord-line y position (vertically centred)

Re    = 100
Nu    = 0.01
U_inf = Re * Nu / C   # Freestream speed (same formula as cylinder)

U = U_inf * np.cos(alpha)   # Inlet x-velocity component
V = U_inf * np.sin(alpha)   # Inlet y-velocity component (from AoA)

# Create the grid
x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
X, Y = np.meshgrid(x, y)

# NACA 4-digit symmetric thickness distribution
def naca_thickness(xn, t):
    xn = np.clip(xn, 0.0, 1.0)
    return 5*t*(0.2969*np.sqrt(xn) - 0.1260*xn - 0.3516*xn**2 + 0.2843*xn**3 - 0.1015*xn**4)

# Normalised chord coordinate for every grid cell
x_norm = (X - x_le) / C
yt     = naca_thickness(x_norm, t_naca)

# Airfoil mask: inside chord range AND within the thickness envelope
airfoil_mask = (x_norm >= 0) & (x_norm <= 1) & (np.abs(Y - y_c) <= yt)

# Velocity and pressure fields
u = np.zeros((Ny, Nx))
v = np.zeros((Ny, Nx))
p = np.zeros((Ny, Nx))

# Stability criterion (same as cylinder)
safety_factor = 0.8
dt = safety_factor / (U_inf/dx + U_inf/dy + 2*Nu/dx**2 + 2*Nu/dy**2)
print(f"Time step (dt) for stability: {dt:.5f} seconds")

CFL_x = U_inf * dt / dx
CFL_y = U_inf * dt / dy
r     = Nu * dt / dx**2

print(f"dt       = {dt:.6f}")
print(f"CFL_x    = {CFL_x:.4f}")
print(f"CFL_y    = {CFL_y:.4f}")
print(f"r        = {r:.4f}")
print(f"Combined = {CFL_x + CFL_y + 4*r:.4f}  (must be \u2264 1)")

nt = 20000
T  = nt * dt
print(f"Total simulation time T = {T:.4f}")

Time step (dt) for stability: 0.03333 seconds
dt       = 0.033333
CFL_x    = 0.3333
CFL_y    = 0.3333
r        = 0.0333
Combined = 0.8000  (must be ≤ 1)
Total simulation time T = 666.6667


## Initial condition

In [3]:
# Set initial conditions — uniform freestream at angle of attack
u = np.full((Ny, Nx), U)
v = np.full((Ny, Nx), V)
u[airfoil_mask] = 0.0   # No-slip on airfoil
v[airfoil_mask] = 0.0

# Algorithm Per Time Step

## Apply Boundary Conditions to u, v

In [4]:
def apply_boundary_conditions(u, v, airfoil_mask):
    # Inlet: uniform freestream with angle of attack
    u[:, 0] = U
    v[:, 0] = V

    # Outlet: zero-gradient (Neumann)
    u[:, -1] = u[:, -2]
    v[:, -1] = v[:, -2]

    # Top and bottom: free-slip
    u[0, :]  = u[1, :]
    v[0, :]  = 0.0
    u[-1, :] = u[-2, :]
    v[-1, :] = 0.0

    # No-slip on airfoil surface
    u[airfoil_mask] = 0.0
    v[airfoil_mask] = 0.0

    return u, v

## Momentum Predictor - Compute u\*, v\* (momentum without pressure)

In [5]:
def momentum_predictor(u, v, p, dx, dy, dt, Nu):
    un = u.copy()
    vn = v.copy()

    u_star = un.copy()
    v_star = vn.copy()

    # Neighbours — interior points only
    u_e = un[1:-1, 2:];   u_w = un[1:-1, :-2]
    u_n = un[2:, 1:-1];   u_s = un[:-2, 1:-1]

    v_e = vn[1:-1, 2:];   v_w = vn[1:-1, :-2]
    v_n = vn[2:, 1:-1];   v_s = vn[:-2, 1:-1]

    # Convection — upwind scheme
    conv_u_x = np.maximum(un[1:-1, 1:-1], 0)*(un[1:-1, 1:-1] - u_w)/dx + np.minimum(un[1:-1, 1:-1], 0)*(u_e - un[1:-1, 1:-1])/dx
    conv_u_y = np.maximum(vn[1:-1, 1:-1], 0)*(un[1:-1, 1:-1] - u_s)/dy + np.minimum(vn[1:-1, 1:-1], 0)*(u_n - un[1:-1, 1:-1])/dy

    conv_v_x = np.maximum(un[1:-1, 1:-1], 0)*(vn[1:-1, 1:-1] - v_w)/dx + np.minimum(un[1:-1, 1:-1], 0)*(v_e - vn[1:-1, 1:-1])/dx
    conv_v_y = np.maximum(vn[1:-1, 1:-1], 0)*(vn[1:-1, 1:-1] - v_s)/dy + np.minimum(vn[1:-1, 1:-1], 0)*(v_n - vn[1:-1, 1:-1])/dy

    conv_u = conv_u_x + conv_u_y
    conv_v = conv_v_x + conv_v_y

    # Diffusion — central differences
    diff_u = Nu*(u_e - 2*un[1:-1, 1:-1] + u_w)/dx**2 + Nu*(u_n - 2*un[1:-1, 1:-1] + u_s)/dy**2
    diff_v = Nu*(v_e - 2*vn[1:-1, 1:-1] + v_w)/dx**2 + Nu*(v_n - 2*vn[1:-1, 1:-1] + v_s)/dy**2

    u_star[1:-1, 1:-1] = un[1:-1, 1:-1] + dt*(diff_u - conv_u)
    v_star[1:-1, 1:-1] = vn[1:-1, 1:-1] + dt*(diff_v - conv_v)

    return u_star, v_star

## Compute divergence b = ∇·u\*

In [6]:
def compute_divergence(u_star, v_star, dx, dy):
    div = (u_star[1:-1, 2:] - u_star[1:-1, :-2]) / (2*dx) \
        + (v_star[2:, 1:-1] - v_star[:-2, 1:-1]) / (2*dy)
    return div

## Connector - Pressure Poisson Solver

In [7]:
def pressure_poisson_connector(u_star, v_star, p, dx, dy, dt, rho=1.0, max_iter=20000, tol=1e-4):
    for it in range(max_iter):
        p_old = p.copy()

        p_e = p_old[1:-1, 2:];  p_w = p_old[1:-1, :-2]
        p_n = p_old[2:, 1:-1];  p_s = p_old[:-2, 1:-1]

        div = compute_divergence(u_star, v_star, dx, dy)
        rhs = (rho / dt) * div

        # Jacobi update
        p[1:-1, 1:-1] = (p_e + p_w)*dy**2 + (p_n + p_s)*dx**2 - rhs*dx**2*dy**2
        p[1:-1, 1:-1] /= (2*(dx**2 + dy**2))

        # Neumann BCs
        p[0, :]  = p[1, :]
        p[-1, :] = p[-2, :]
        p[:, 0]  = p[:, 1]
        p[:, -1] = p[:, -2]

        p[:, -1] = 0.0  # Dirichlet reference at outlet

        if np.linalg.norm(p - p_old) < tol:
            break
    else:
        print("Pressure Poisson did not converge within the maximum iterations.")

    return p

## Correct: u = u\* - dt·∂p/∂x,  v = v\* - dt·∂p/∂y

In [8]:
def velocity_corrector(u_star, v_star, p, dx, dy, dt, rho=1.0):
    u = u_star.copy()
    v = v_star.copy()

    p_e = p[1:-1, 2:];  p_w = p[1:-1, :-2]
    p_n = p[2:, 1:-1];  p_s = p[:-2, 1:-1]

    u[1:-1, 1:-1] = u_star[1:-1, 1:-1] - (dt/rho)*(p_e - p_w)/(2*dx)
    v[1:-1, 1:-1] = v_star[1:-1, 1:-1] - (dt/rho)*(p_n - p_s)/(2*dy)

    return u, v

## Track ${max|u^{n+1} - u^n|}$ for steady state convergence

In [9]:
def check_convergence(u, v, u_old, v_old, tol=1e-4):
    du = np.linalg.norm(u - u_old)
    dv = np.linalg.norm(v - v_old)
    return du < tol and dv < tol

## Compute Drag and Lift

In [10]:
def compute_forces(u, v, airfoil_mask, dx, dy, rho=1.0):
    # Integrated momentum flux at airfoil cells (same approach as cylinder)
    Fx = -rho * np.sum(u[airfoil_mask]) * dx * dy / dt
    Fy = -rho * np.sum(v[airfoil_mask]) * dx * dy / dt
    # Rotate body-axis forces into wind-axis (drag along flow, lift perpendicular)
    Fd =  Fx * np.cos(alpha) + Fy * np.sin(alpha)
    Fl = -Fx * np.sin(alpha) + Fy * np.cos(alpha)
    Cd = Fd / (0.5 * rho * U_inf**2 * C)
    Cl = Fl / (0.5 * rho * U_inf**2 * C)
    return Cd, Cl

## Main Loop

In [11]:
residual_history = []
u_history = []
v_history = []
p_history = []
drag_history = []
lift_history = []

for n in range(nt):
    u_old = u.copy()
    v_old = v.copy()
    p_old = p.copy()

    u_old, v_old = apply_boundary_conditions(u_old, v_old, airfoil_mask)
    u_star, v_star = momentum_predictor(u_old, v_old, p_old, dx, dy, dt, Nu)
    p = pressure_poisson_connector(u_star, v_star, p_old, dx, dy, dt)
    u, v = velocity_corrector(u_star, v_star, p, dx, dy, dt)

    drag_coeff, lift_coeff = compute_forces(u, v, airfoil_mask, dx, dy)
    drag_history.append(drag_coeff)
    lift_history.append(lift_coeff)

    u, v = apply_boundary_conditions(u, v, airfoil_mask)

    if n % 50 == 0:
        u_history.append(u.copy())
        v_history.append(v.copy())
        p_history.append(p.copy())

    residual = np.max(np.abs(u - u_old))
    residual_history.append(residual)
    if n % 100 == 0:
        print(f"Step {n:4d} | residual = {residual:.2e}")

Pressure Poisson did not converge within the maximum iterations.
Step    0 | residual = 3.35e-01
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum iterations.
Pressure Poisson did not converge within the maximum itera

## Result Visualization

In [12]:
# Helper: build an airfoil outline polygon for plotting
def airfoil_patch(color='gray'):
    xn = np.linspace(0, 1, 300)
    yt_line = naca_thickness(xn, t_naca)
    xs = np.concatenate([xn, xn[::-1]])
    ys = np.concatenate([yt_line, -yt_line[::-1]])
    verts = np.column_stack([x_le + xs*C, y_c + ys])
    return plt.Polygon(verts, color=color, zorder=5)

In [ ]:
# Velocity Field (quiver)
step  = 2
speed = np.sqrt(u[::step, ::step]**2 + v[::step, ::step]**2)

plt.figure(figsize=(12, 6))
plt.title(f"Velocity Field  (NACA 0012, Re={Re}, AoA={AoA}°)")
q = plt.quiver(
    X[::step, ::step], Y[::step, ::step],
    u[::step, ::step], v[::step, ::step],
    speed, cmap='jet', clim=[0, speed.max()]
)
plt.colorbar(q, label='Speed |u|')
plt.gca().add_patch(airfoil_patch())
plt.xlabel('x'); plt.ylabel('y')
plt.xlim(0, Lx); plt.ylim(0, Ly)
plt.tight_layout()
plt.show()

In [ ]:
# Streamlines
speed = np.sqrt(u**2 + v**2)
plt.figure(figsize=(12, 6))
plt.title(f"Streamlines  (NACA 0012, Re={Re}, AoA={AoA}°)")
plt.streamplot(X, Y, u, v, density=2, color=speed, cmap='plasma', linewidth=1)
plt.gca().add_patch(airfoil_patch())
plt.colorbar(label='Speed')
plt.xlabel('x'); plt.ylabel('y')
plt.xlim(0, Lx); plt.ylim(0, Ly)
plt.tight_layout()
plt.show()

In [ ]:
# Velocity Magnitude Contour
speed = np.sqrt(u**2 + v**2)
plt.figure(figsize=(12, 6))
plt.title(f"Velocity Magnitude  (NACA 0012, Re={Re}, AoA={AoA}°)")
plt.contourf(X, Y, speed, levels=50, cmap='jet')
plt.gca().add_patch(airfoil_patch())
plt.colorbar(label='Speed')
plt.xlabel('x'); plt.ylabel('y')
plt.xlim(0, Lx); plt.ylim(0, Ly)
plt.tight_layout()
plt.show()

In [ ]:
# Pressure Contour
plt.figure(figsize=(12, 6))
plt.title(f"Pressure Field  (NACA 0012, Re={Re}, AoA={AoA}°)")
plt.contourf(X, Y, p, levels=50, cmap='viridis', alpha=0.8)
plt.gca().add_patch(airfoil_patch())
plt.colorbar(label='Pressure')
plt.xlabel('x'); plt.ylabel('y')
plt.xlim(0, Lx); plt.ylim(0, Ly)
plt.tight_layout()
plt.show()

In [ ]:
# Vorticity Contour: omega = dv/dx - du/dy
omega = (v[1:-1, 2:] - v[1:-1, :-2]) / (2*dx) \
      - (u[2:, 1:-1] - u[:-2, 1:-1]) / (2*dy)

vlim = np.max(np.abs(omega))
plt.figure(figsize=(12, 6))
plt.title(f"Vorticity  (NACA 0012, Re={Re}, AoA={AoA}°)  — red=CCW, blue=CW")
plt.contourf(X[1:-1, 1:-1], Y[1:-1, 1:-1], omega,
             levels=100, cmap='seismic', vmin=-vlim, vmax=vlim)
plt.gca().add_patch(airfoil_patch())
plt.colorbar(label='Vorticity')
plt.xlabel('x'); plt.ylabel('y')
plt.xlim(2, 18); plt.ylim(2, 8)
plt.tight_layout()
plt.show()

In [ ]:
# Drag Coefficient History
plt.figure(figsize=(12, 6))
plt.plot(drag_history)
plt.xlabel('Time step'); plt.ylabel('Cd')
plt.title(f"Drag Coefficient History  (NACA 0012, Re={Re}, AoA={AoA}°)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Lift Coefficient History
plt.figure(figsize=(12, 6))
plt.plot(lift_history, color='tomato')
plt.xlabel('Time step'); plt.ylabel('Cl')
plt.title(f"Lift Coefficient History  (NACA 0012, Re={Re}, AoA={AoA}°)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Centreline Profiles at y = y_c (chord line, mid-row)
mid = Ny // 2

# U-velocity along chord line
plt.figure(figsize=(12, 6))
plt.plot(x, u[mid, :], color='steelblue')
plt.axvline(x_le,     color='gray',  linestyle='--', label='Leading edge')
plt.axvline(x_le + C, color='black', linestyle='--', label='Trailing edge')
plt.title(f"Centreline u-velocity at y = {y_c}  (Re={Re}, AoA={AoA}°)")
plt.xlabel('x'); plt.ylabel('u  (x-velocity)')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

# V-velocity along chord line
plt.figure(figsize=(12, 6))
plt.plot(x, v[mid, :], color='darkorange')
plt.axvline(x_le,     color='gray',  linestyle='--', label='Leading edge')
plt.axvline(x_le + C, color='black', linestyle='--', label='Trailing edge')
plt.title(f"Centreline v-velocity at y = {y_c}  (Re={Re}, AoA={AoA}°)")
plt.xlabel('x'); plt.ylabel('v  (y-velocity)')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

# Pressure along chord line
plt.figure(figsize=(12, 6))
plt.plot(x, p[mid, :], color='seagreen')
plt.axvline(x_le,     color='gray',  linestyle='--', label='Leading edge')
plt.axvline(x_le + C, color='black', linestyle='--', label='Trailing edge')
plt.title(f"Centreline Pressure at y = {y_c}  (Re={Re}, AoA={AoA}°)")
plt.xlabel('x'); plt.ylabel('p  (pressure)')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Residual History
plt.figure(figsize=(12, 6))
plt.semilogy(residual_history)
plt.title(f"Residual History  (NACA 0012, Re={Re}, AoA={AoA}°)")
plt.xlabel('Time step')
plt.ylabel('Max |u - u_old|')
plt.grid(True)
plt.tight_layout()
plt.show()

In [21]:
# Lift oscillation frequency (analogous to Strouhal analysis for cylinder)
from scipy.signal import find_peaks

lift_array = np.array(lift_history)
peaks, _   = find_peaks(lift_array, height=0)

if len(peaks) > 1:
    shedding_period    = np.mean(np.diff(peaks)) * dt
    shedding_frequency = 1 / shedding_period
    St = shedding_frequency * C / U_inf
    print(f"Lift oscillation frequency f = {shedding_frequency:.4f} Hz")
    print(f"Strouhal number St = {St:.4f}")
    print(f"Final Cd = {drag_history[-1]:.4f},  Final Cl = {lift_history[-1]:.4f}")
else:
    print("Steady flow: no oscillation detected in lift coefficient.")
    print(f"Final Cd = {drag_history[-1]:.4f},  Final Cl = {lift_history[-1]:.4f}")

Lift oscillation frequency f = 0.0523 Hz
Strouhal number St = 0.0523
Final Cd = -0.3641,  Final Cl = 0.0241


## Export to ParaView (VTK ImageData)

Writes `.vti` files (VTK ImageData) readable by ParaView with four fields:

| Field | Description |
|---|---|
| `Velocity` | 3-component vector (u, v, 0) |
| `Pressure` | Scalar pressure field |
| `Vorticity` | Scalar ω = ∂v/∂x − ∂u/∂y |
| `AirfoilMask` | 1 inside airfoil, 0 in fluid |

**Outputs**
- `paraview_export_airfoil/airfoil_flow_final.vti` — final snapshot
- `paraview_export_airfoil/airfoil_flow.pvd` — full time series (open this in ParaView)

In [22]:
import os, struct, base64

output_dir    = "paraview_export_airfoil"
export_stride = 10   # export every 10th history snapshot (= every 500 time steps)

os.makedirs(output_dir, exist_ok=True)

def _b64(arr_flat):
    """VTK binary base64: UInt32 byte-count header + raw float64 data."""
    raw = np.asarray(arr_flat, dtype=np.float64).tobytes()
    return base64.b64encode(struct.pack('<I', len(raw)) + raw).decode()

def write_vti(filepath, u_a, v_a, p_a):
    """Write one VTK ImageData (.vti) file for ParaView."""
    Ny, Nx = u_a.shape

    omega = np.zeros((Ny, Nx))
    omega[1:-1, 1:-1] = (
        (v_a[1:-1, 2:] - v_a[1:-1, :-2]) / (2*dx)
      - (u_a[2:, 1:-1] - u_a[:-2, 1:-1]) / (2*dy)
    )

    vel_flat  = np.column_stack([u_a.ravel(), v_a.ravel(),
                                 np.zeros(Nx*Ny)]).ravel()
    mask_flat = airfoil_mask.ravel().astype(np.float64)

    fields = [
        ("Velocity",    3, vel_flat),
        ("Pressure",    1, p_a.ravel()),
        ("Vorticity",   1, omega.ravel()),
        ("AirfoilMask", 1, mask_flat),
    ]

    with open(filepath, 'w') as f:
        f.write('<?xml version="1.0"?>\n')
        f.write('<VTKFile type="ImageData" version="0.1" '
                'byte_order="LittleEndian" header_type="UInt32">\n')
        f.write(f'  <ImageData WholeExtent="0 {Nx-1} 0 {Ny-1} 0 0" '
                f'Origin="0 0 0" Spacing="{dx} {dy} 1.0">\n')
        f.write(f'    <Piece Extent="0 {Nx-1} 0 {Ny-1} 0 0">\n')
        f.write('      <PointData Scalars="Pressure" Vectors="Velocity">\n')

        for name, nc, data in fields:
            nc_attr = f'NumberOfComponents="{nc}" ' if nc > 1 else ''
            f.write(f'        <DataArray type="Float64" Name="{name}" '
                    f'{nc_attr}format="binary">\n')
            f.write(f'          {_b64(data)}\n')
            f.write('        </DataArray>\n')

        f.write('      </PointData>\n')
        f.write('    </Piece>\n')
        f.write('  </ImageData>\n')
        f.write('</VTKFile>\n')

# 1. Final snapshot
write_vti(os.path.join(output_dir, "airfoil_flow_final.vti"), u, v, p)
print("\u2713 Final snapshot \u2192 paraview_export_airfoil/airfoil_flow_final.vti")

# 2. Time series + PVD collection
snapshot_indices = range(0, len(u_history), export_stride)
pvd_rows = []
for k in snapshot_indices:
    sim_time = k * 50 * dt
    fname    = f"airfoil_flow_{k:04d}.vti"
    write_vti(os.path.join(output_dir, fname), u_history[k], v_history[k], p_history[k])
    pvd_rows.append((sim_time, fname))

pvd_path = os.path.join(output_dir, "airfoil_flow.pvd")
with open(pvd_path, 'w') as f:
    f.write('<?xml version="1.0"?>\n')
    f.write('<VTKFile type="Collection" version="0.1">\n')
    f.write('  <Collection>\n')
    for t, fname in pvd_rows:
        f.write(f'    <DataSet timestep="{t:.4f}" file="{fname}"/>\n')
    f.write('  </Collection>\n')
    f.write('</VTKFile>\n')

print(f"\u2713 {len(pvd_rows)} time steps \u2192 paraview_export_airfoil/airfoil_flow.pvd")
print()
print("\u2500\u2500 ParaView workflow \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
print("  1. File \u2192 Open \u2192 airfoil_flow.pvd   (loads full time series)")
print("     OR  airfoil_flow_final.vti        (single snapshot)")
print("  2. Press Apply, then click the '2D' button for top-down view")
print()
print("  Useful filters:")
print("  \u2022 Color by Vorticity + RdBu colormap  \u2192 see shear layers & wake")
print("  \u2022 Filters \u2192 Glyph (Velocity, Arrow)   \u2192 velocity vectors")
print("  \u2022 Filters \u2192 Stream Tracer (Velocity)  \u2192 streamlines")
print("  \u2022 Filters \u2192 Threshold (AirfoilMask > 0.5) \u2192 isolate airfoil")

✓ Final snapshot → paraview_export_airfoil/airfoil_flow_final.vti
✓ 40 time steps → paraview_export_airfoil/airfoil_flow.pvd

── ParaView workflow ───────────────────────────────────────────
  1. File → Open → airfoil_flow.pvd   (loads full time series)
     OR  airfoil_flow_final.vti        (single snapshot)
  2. Press Apply, then click the '2D' button for top-down view

  Useful filters:
  • Color by Vorticity + RdBu colormap  → see shear layers & wake
  • Filters → Glyph (Velocity, Arrow)   → velocity vectors
  • Filters → Stream Tracer (Velocity)  → streamlines
  • Filters → Threshold (AirfoilMask > 0.5) → isolate airfoil
